# 📈 Correlación y regresión lineal simple · Resuelto

**Métodos Cuantitativos II · Antropología UAH · 11 de diciembre de 2026**

## 🌍 Antes de abrir la base: ¿qué queremos investigar?

Hacia 1888, Suiza atravesaba una **transición demográfica**: la fecundidad comenzaba a descender, pero los cambios no ocurrían del mismo modo en todos los territorios. Esta base reúne datos de **47 distritos francófonos** y permite preguntar: **¿cómo se relacionaban las diferencias de fecundidad entre distritos con sus condiciones sociales y económicas?** [Fuente y descripción de la base](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/swiss.html).

La mirada sociológica consiste en poner las diferencias demográficas en contexto. Por ejemplo, los distritos podían diferir en el acceso a la educación y en sus formas de trabajo. Si vemos que esas características van de la mano con la fecundidad, tenemos una pista para investigar cómo se organizaban la vida familiar y el trabajo en esos lugares. El gráfico por sí solo no explica por qué aparece el patrón.

| Variables que cruzaremos | ¿Para qué sirve compararlas? |
|:---|:---|
| **Escolaridad y fecundidad** | Para ver si los distritos con mayor proporción de reclutas que estudiaron más allá de la primaria tienden a presentar mayor o menor fecundidad. La escolaridad medida es la de los reclutas, no la de toda la población. |
| **Trabajo agrícola y fecundidad** | Para ver si el patrón de fecundidad cambia entre distritos donde una proporción mayor o menor de los hombres se dedica a la agricultura. Podemos pensar si la organización del trabajo y de los hogares ayuda a entender ese patrón. |
| **Mortalidad infantil y fecundidad** | Para explorar si los distritos donde mueren más niños antes de cumplir un año también muestran diferencias de fecundidad. Podemos preguntarnos si condiciones compartidas, como el saneamiento o el acceso a cuidados, se relacionan con ambos indicadores. |

Aquí **cruzar** significa mirar **dos variables juntas para los mismos distritos**. Primero ubicamos cada distrito como un punto, luego resumimos la asociación y, por último, estimamos una recta para expresar cuánto cambia la fecundidad ajustada al comparar distritos con distintos valores del predictor.

📌 **Qué podemos concluir:** hablamos de diferencias **entre distritos**, no de lo que hizo cada persona o familia. Tampoco podemos afirmar que la escolaridad, la agricultura o la mortalidad infantil causaron un cambio en la fecundidad. Para investigar explicaciones harían falta otras fuentes, como censos, registros históricos o estudios sobre la vida familiar de la época.

💬 **Antes de programar:** elige uno de los cruces. ¿Qué patrón esperarías ver? ¿Qué otra característica del distrito buscarías para interpretar ese resultado?

Al terminar podrás describir una asociación con un gráfico, comparar tres coeficientes de correlación, interpretar una regresión lineal y distinguir una predicción dentro del rango de los datos de una extrapolación.

🧭 **Cómo trabajar:** ejecuta los bloques en orden. Antes de cada bloque de análisis, escribe qué esperas observar; después, explica el resultado con palabras y unidades. En Colab, usa un entorno de ejecución **R**.

## 📂 0. Subir el archivo de datos

1. Ten a mano el archivo **`swiss.csv`** entregado para esta clase.
2. En Colab, pulsa **Conectar** y abre **Archivos** (ícono de carpeta a la izquierda).
3. Pulsa **Subir al almacenamiento de la sesión** y selecciona `swiss.csv`. Comprueba que aparezca en el panel con ese nombre.
4. Ejecuta la primera celda de R. Deberías ver seis filas de la base. Si inicias una nueva sesión y el archivo ya no aparece, vuelve a subirlo.

## 🔎 1. Conocer los datos

Cada fila representa uno de **47 distritos francófonos de Suiza** hacia **1888**. `Canton` conserva el nombre de la columna del CSV original; aquí identifica al distrito. Los datos son agregados: un patrón entre distritos no describe necesariamente lo que ocurre entre personas.

| Variable | Qué mide |
|:---|:---|
| `Fertility` | Índice estandarizado de fecundidad; no es un número de hijos ni un porcentaje. |
| `Education` | Porcentaje de reclutas con educación posterior a la primaria. |
| `Agriculture` | Porcentaje de hombres ocupados en agricultura. |
| `Catholic` | Porcentaje de población católica. |
| `Examination` | Porcentaje de reclutas con la mejor calificación en el examen militar. |
| `InfantMortality` | Porcentaje de nacidos vivos que fallecieron antes de cumplir un año. |

In [ ]:
# Importamos el CSV que subiste al almacenamiento de la sesión.
archivo <- if (dir.exists("/content")) "/content/swiss.csv" else "swiss.csv"
if (!file.exists(archivo)) {
  stop("No encuentro swiss.csv. Súbelo desde Archivos en Colab y vuelve a ejecutar esta celda.", call. = FALSE)
}
suiza <- read.csv(archivo, stringsAsFactors = FALSE)
columnas <- c("Canton", "Fertility", "Agriculture", "Examination", "Education", "Catholic", "InfantMortality")
if (!all(columnas %in% names(suiza))) {
  stop("El archivo no tiene las columnas esperadas. Revisa que sea el swiss.csv de la clase.", call. = FALSE)
}

# Si ggplot2 ya está disponible, este bloque no instala nada.
if (!requireNamespace("ggplot2", quietly = TRUE)) {
  install.packages("ggplot2", repos = "https://cloud.r-project.org")
}
library(ggplot2)
theme_set(theme_minimal(base_size = 13))

head(suiza)

In [ ]:
# dim() cuenta filas y columnas; summary() resume los valores observados.
# Min. y Max. muestran el valor más bajo y más alto de cada variable.

dim(suiza)
summary(suiza[c("Education", "Fertility", "Agriculture")])
colSums(is.na(suiza))
# Rango de escolaridad: el primer número es el mínimo; el segundo, el máximo.
range(suiza$Education)

🔎 **¿Qué nos dice el rango?** `Education` es el porcentaje de reclutas con educación posterior a primaria en cada distrito. El mínimo y el máximo son los porcentajes más bajo y más alto que sí observamos. Sirven para describir los datos y reconocer si una comparación queda fuera de lo observado. No hace falta memorizar esos números.

## 👀 2. Mirar antes de calcular

Cada punto es un distrito. Ponemos la escolaridad en el eje horizontal y la fecundidad en el vertical. Lee la nube de izquierda a derecha: ¿sube o baja? ¿Se parece a una recta? ¿Hay puntos apartados?

In [ ]:
# Cada punto es un distrito. X = escolaridad; Y = fecundidad.
# Lee primero la dirección de la nube: ¿tiende a subir o a bajar?

nube <- ggplot(suiza, aes(x = Education, y = Fertility)) +
  labs(x = "Reclutas con educación posterior a primaria (%)",
       y = "Índice estandarizado de fecundidad")
nube + geom_point(color = "#176B73", size = 2.5) +
  labs(title = "Escolaridad y fecundidad por distrito")

In [ ]:
# Catholic cambia el tamaño de los puntos; seguimos usando los mismos ejes.
# Los nombres se muestran solo para algunos distritos para facilitar la lectura.

destacados <- subset(suiza, Education >= 30)
nube + geom_point(aes(size = Catholic), color = "#176B73", alpha = 0.7) +
  geom_text(data = destacados, aes(label = Canton),
            nudge_y = 3, check_overlap = TRUE, size = 3) +
  scale_size_area(max_size = 7) +
  labs(title = "Una tercera variable: composición religiosa",
       size = "Población católica (%)")

Cambiar el tamaño de los puntos ayuda a formular preguntas sobre una tercera variable. No equivale a controlarla estadísticamente. ¿Qué procesos históricos podrían relacionar educación y fecundidad? ¿Qué datos necesitarías para evaluarlos?

## 📊 3. Medir la asociación

- **Pearson (`r`)** resume qué tan alineados están los puntos con una recta. Puede ser sensible a observaciones extremas.
- **Spearman (`rho`)** aplica la correlación a los rangos. Puede detectar una tendencia que siempre sube o baja aunque no sea recta.
- **Kendall (`tau`)** compara pares: si dos distritos se ordenan en el mismo sentido en ambas variables, el par es concordante; si se ordenan al revés, es discordante.

Los tres van de −1 a 1. El signo indica la dirección. Un valor cercano a cero indica poca asociación del tipo que mide ese coeficiente. Sus magnitudes no son directamente intercambiables.

In [ ]:
# cor() devuelve el coeficiente, sin prueba ni intervalo.
# Compara el signo; las magnitudes de los tres métodos no son equivalentes.

metodos <- c("pearson", "spearman", "kendall")
coeficientes <- sapply(metodos, function(metodo) {
  cor(suiza$Education, suiza$Fertility, method = metodo)
})
round(coeficientes, 3)

💬 **Lectura guiada:** los tres signos son negativos. Pearson da aproximadamente −0,664; los coeficientes por rangos también muestran una tendencia descendente. El gráfico ayuda a entender por qué el orden y la alineación lineal no cuentan exactamente la misma historia.

### 🧪 ¿Qué aporta una prueba?

Para Pearson, la hipótesis nula plantea que la correlación lineal poblacional es cero bajo el modelo de la prueba. Lee el coeficiente, su intervalo de confianza y el valor p por separado. En Spearman y Kendall usamos `exact = FALSE` para calcular valores p aproximados con empates.

In [ ]:
# cor.test() muestra el coeficiente y el valor p de la prueba bilateral.
# En Pearson, el intervalo que aparece es el de la correlación, no el de la pendiente.
# exact = FALSE pide un valor p aproximado para las pruebas por rangos con empates.

prueba_pearson <- cor.test(~ Education + Fertility, data = suiza, method = "pearson")
prueba_spearman <- cor.test(~ Education + Fertility, data = suiza,
                            method = "spearman", exact = FALSE)
prueba_kendall <- cor.test(~ Education + Fertility, data = suiza,
                           method = "kendall", exact = FALSE)
print(prueba_pearson)
print(prueba_spearman)
print(prueba_kendall)

### 🧾 Cómo leer la salida de las tres pruebas

Empieza por **sample estimates**: ahí está el coeficiente. Luego mira **p-value** para evaluar la hipótesis nula de que el coeficiente poblacional correspondiente es cero. La línea **alternative hypothesis** dice que la prueba es bilateral: busca una asociación distinta de cero en cualquiera de las dos direcciones.

| Prueba | Coeficiente estimado | Valor p | ¿Qué vemos? |
|:---|---:|---:|:---|
| Pearson | −0,664 | 0,0000003659 | La asociación lineal estimada es negativa. |
| Spearman | −0,443 | 0,001806 | Al ordenar los distritos, mayor escolaridad tiende a acompañarse de menor fecundidad. |
| Kendall | −0,331 | 0,001394 | Al comparar pares, predominan los que quedan en orden opuesto en ambas variables. |

Los tres valores p son menores que 0,05. Bajo los supuestos de las pruebas, cada resultado aporta evidencia contra su hipótesis de coeficiente cero. **El valor p no es la probabilidad de que la hipótesis sea cierta ni mide la fuerza de la asociación.** Tampoco elegimos una medida por tener el menor valor p.

**¿Qué son `t`, `S` y `z`?** Son los estadísticos que R usa para calcular el valor p de cada prueba; no son los coeficientes. En Pearson aparecen 45 grados de libertad: 47 distritos menos 2. En Spearman y Kendall el valor p es aproximado porque usamos `exact = FALSE` con datos que tienen empates.

**¿Dónde aparece el intervalo del 95 %?** Esta salida de `cor.test()` lo muestra para **Pearson**: [−0,799; −0,465]. Todo el intervalo queda bajo cero, de acuerdo con la prueba bilateral al 5 %. Se refiere a la **correlación**, que no tiene unidades. Para Spearman y Kendall, esta función no muestra un intervalo en la salida estándar.

🗣️ **Discute:** un valor p pequeño no mide la fuerza de la relación ni identifica una causa. La interpretación inferencial requiere supuestos como independencia entre observaciones; distritos vecinos podrían estar relacionados. Estos resultados son un ejercicio con datos históricos, no una generalización a cualquier lugar o época.

## 📐 4. Ajustar una regresión lineal simple

La correlación es simétrica. Para predecir con una regresión, en cambio, elegimos un resultado ($Y$) y un predictor ($X$). En `lm(Y ~ X)` la variable a la izquierda es la que queremos explicar o predecir. Mínimos cuadrados elige la recta que minimiza la suma de los residuos al cuadrado. Cada residuo es el valor observado menos el ajustado.

### 📍 Antes de ajustar: ¿qué es un residuo?

Imagina que un distrito registra **60 puntos** de fecundidad y la recta estima **70**. Comparamos dos valores para el **mismo distrito** y la **misma escolaridad**: el punto observado y el punto de la recta. La diferencia vertical entre ellos, con signo, es el **residuo**.

| Residuo = observado − ajustado | ¿Dónde queda el punto observado? | ¿Qué hizo la recta? |
|:---|:---|:---|
| **Positivo** | Por encima de la recta | Estimó de menos. |
| **Negativo** | Por debajo de la recta | Estimó de más. |
| **Cero** | Sobre la recta | Acertó el valor observado. |

El signo indica **en qué dirección** se apartó la estimación; la distancia a cero indica **por cuántos puntos**. Ahora probemos con los números del ejemplo:

In [ ]:
# Ejemplo inventado: el dato observado fue 60, pero la recta estimó 70.
observado_ejemplo <- 60
ajustado_ejemplo <- 70
observado_ejemplo - ajustado_ejemplo
# Da -10: la recta estimó 10 puntos de más.
# Antes de seguir: si intercambiamos 60 y 70, ¿qué signo tendría el residuo?

In [ ]:
# En lm(Y ~ X), Y va a la izquierda y X a la derecha.
# summary(): lee la pendiente en la fila Education, columna Estimate.
# confint(): busca los límites del 95 % en la misma fila. ¿Queda el cero entre ellos?

modelo_educacion <- lm(Fertility ~ Education, data = suiza)
print(summary(modelo_educacion))
print(confint(modelo_educacion))

La recta estimada es $\widehat{Y}=79.61-0.862X$.

- **Pendiente:** un punto porcentual adicional de educación se asocia con **0,86 puntos menos** en el índice de fecundidad ajustado. No es una reducción del 0,86 % ni un efecto causal.
- **Intercepto:** 79,61 es el valor que la recta predice para `Education = 0`, apenas fuera del rango observado. Evita darle una interpretación sustantiva automática.
- **$R^2$:** el modelo da aproximadamente 0,44. La recta representa un 44 % de la variación observada de fecundidad en estos datos.

### 🎯 ¿Qué significa el intervalo de la pendiente?

`confint(modelo_educacion)` devuelve los límites del intervalo del 95 % para cada coeficiente. Lee la fila **Education**, no la de **(Intercept)**:

| Pendiente estimada | Límite inferior | Límite superior |
|---:|---:|---:|
| −0,86 | −1,15 | −0,57 |

Por cada **punto porcentual adicional** de escolaridad, la recta estima **0,86 puntos menos** en el índice de fecundidad. Bajo los supuestos del modelo, los datos son compatibles con una disminución de **0,57 a 1,15 puntos** por punto porcentual adicional.

- Si el intervalo **excluye cero**, se rechaza una pendiente nula al 5 % con la prueba bilateral correspondiente.
- Si el intervalo **incluye cero**, no podemos descartar una pendiente nula. Eso no demuestra que no exista una relación.
- Este intervalo es para la **pendiente**. No es el intervalo de la correlación de Pearson ni un intervalo para el valor de un distrito nuevo.

In [ ]:
# La línea es el valor ajustado por el modelo para cada nivel de escolaridad.
# La banda corresponde a la media estimada, no a un distrito individual.

nube + geom_point(color = "#176B73", size = 2.5) +
  geom_smooth(method = "lm", formula = y ~ x, color = "#A14225",
              fill = "#F1C4AE", se = TRUE) +
  labs(title = "Recta estimada con escolaridad",
       subtitle = "La banda estima la incertidumbre de la media ajustada")

### 📍 Un residuo es la distancia vertical entre el dato y la recta

**Residuo = valor observado − valor ajustado.** Para Courtelary, la fecundidad observada es **80,20** y la recta estima **69,26**. El residuo es **+10,94** puntos: la recta **subestima** el valor observado. Si el residuo fuera negativo, la recta lo **sobreestimaría**.

In [ ]:
# fitted() da lo que estima la recta; residuals() resta ese valor al observado.
# Conservamos el nombre del distrito para saber qué caso estamos mirando.

residuos_distritos <- data.frame(
  distrito = suiza$Canton,
  observado = suiza$Fertility,
  ajustado = round(fitted(modelo_educacion), 2),
  residuo = round(residuals(modelo_educacion), 2)
)
head(residuos_distritos)

In [ ]:
# Eje X: fecundidad ajustada. Eje Y: observado menos ajustado.
# Arriba de cero: la recta quedó corta. Abajo: la recta estimó de más.

plot(modelo_educacion, which = 1)

**Cómo leer el gráfico anterior:** a la derecha están los valores ajustados más altos; arriba de la línea horizontal están los residuos positivos y abajo, los negativos. Un residuo de cero significa que el punto cae sobre la recta. La línea suave ayuda a detectar un patrón:

- Una **curva** sugiere que la recta deja fuera parte de la forma de la relación.
- Un **abanico** sugiere que los errores son más dispersos en una parte del gráfico.
- Un punto **muy separado de cero** merece revisión; por sí solo no es motivo para borrarlo.

En `summary()`, el bloque **Residuals** muestra mínimo, cuartiles, mediana y máximo. Resume los errores en las **mismas unidades de la fecundidad**, pero no muestra dónde aparecen; por eso también miramos el gráfico.

### 🌾 Cambiar el predictor

Ahora usamos agricultura para explicar la fecundidad. Compara el signo de la pendiente y el R² con el modelo anterior.

In [ ]:
# Cambiamos el predictor a Agriculture, manteniendo Fertility como resultado.
# Lee Residuals para ver cuánto se apartan los distritos de esta recta.
# La pendiente y su intervalo están en la fila Agriculture, no en (Intercept).

modelo_agricultura <- lm(Fertility ~ Agriculture, data = suiza)
summary(modelo_agricultura)
confint(modelo_agricultura)

### 🧾 Cómo leer la salida de agricultura

**Call** confirma qué ajustamos: fecundidad según el porcentaje de hombres ocupados en agricultura. En **Residuals** cada número resume diferencias entre lo observado y lo ajustado, en puntos del índice de fecundidad:

| Resultado | Lectura |
|:---|:---|
| **Min = −25,54** | En el caso con el residuo más negativo, la recta estimó 25,54 puntos **de más**. |
| **1Q = −7,87; 3Q = 9,05** | Aproximadamente la mitad central de los residuos cae entre esos dos valores. |
| **Median = −0,64** | El residuo central está cerca de cero. Eso no significa que todos los distritos queden cerca de la recta. |
| **Max = +24,49** | En el caso con el residuo más positivo, la recta estimó 24,49 puntos **de menos**. |
| **Residual standard error = 11,82** | Resume la dispersión de los residuos alrededor de cero; está en puntos del índice. No es el error máximo ni el error promedio absoluto. |

**Pendiente de Agriculture:** por un punto porcentual adicional en agricultura, el índice ajustado aumenta **0,19 puntos**. El intervalo del 95 % va de **0,04 a 0,35** y excluye cero; el valor p de **0,0149** es coherente con eso. **R² = 0,125**: la recta recoge cerca del 12,5 % de la variación observada. Una pendiente distinta de cero puede coexistir con residuos grandes; no demuestra causalidad.

In [ ]:
# Identifica los distritos de los extremos del bloque Residuals.
# Recuerda: un signo negativo significa que el valor ajustado quedó por encima del observado.
e_agricultura <- residuals(modelo_agricultura)
extremos <- c(which.min(e_agricultura), which.max(e_agricultura))
data.frame(
  distrito = suiza$Canton[extremos],
  observado = suiza$Fertility[extremos],
  ajustado = round(fitted(modelo_agricultura)[extremos], 2),
  residuo = round(e_agricultura[extremos], 2)
)

In [ ]:
# X: fecundidad ajustada. Y: observado menos ajustado.
# La línea horizontal marca cero. Los puntos lejanos son distritos donde la recta falla más.
plot(modelo_agricultura, which = 1)
# Compara con el modelo de escolaridad: ¿la nube de residuos tiene una forma distinta?
# Los números junto a algunos puntos son filas de la base, no valores de fecundidad.

⏸️ **Pausa:** compara el signo de las pendientes y los $R^2$. ¿Cuál de las dos rectas representa más variación en estos datos? Recuerda que son dos modelos separados: ninguno ajusta por el predictor del otro.

### 🧪 Extensión opcional: ¿cuándo extrapolamos?

Este bloque se puede saltar en clase. Sirve para ver que una recta puede dar un número aunque no tengamos datos para respaldarlo. Compara 10 %, 50 % y 90 % con el rango de escolaridad que obtuviste con `range()`. ¿Cuál queda fuera?

In [ ]:
# Rango de los datos: de 1% a 53% de escolaridad.
rango <- range(suiza$Education)
# 10% y 50% están dentro; 90% está fuera y se marca como extrapolación.
nuevos <- data.frame(Education = c(10, 50, 90))
nuevos$extrapolacion <- nuevos$Education < rango[1] | nuevos$Education > rango[2]
nuevos$fecundidad_ajustada <- round(predict(modelo_educacion, newdata = nuevos), 2)
print(nuevos)
# R calcula un número para 90%, pero el patrón podría cambiar fuera de los datos.

## 🤝 Actividad en duplas

**Pregunta:** ¿cómo se relaciona `InfantMortality` con `Fertility`? En cada celda, el ejemplo con `#` muestra cómo se hizo el paso anterior con escolaridad. Úsalo como referencia para trabajar con mortalidad infantil. La actividad no requiere hacer predicciones.

### 📈 1. Dibuja la nube

Pon la mortalidad infantil en el eje X y la fecundidad en el eje Y. Describe si la nube sube o baja, qué forma tiene y si hay algún punto que te llame la atención.

In [ ]:
# Ejemplo anterior: ggplot(suiza, aes(x = Education, y = Fertility)) + geom_point()
# Ahora grafica mortalidad infantil en X y fecundidad en Y.

ggplot(suiza, aes(InfantMortality, Fertility)) +
  geom_point(colour = "#176B73") +
  labs(x = "Mortalidad infantil (%)", y = "Índice de fecundidad")
# Se ve una tendencia ascendente con dispersión.
# Revisa el punto de mortalidad baja: ¿qué distrito es y qué tan aislado queda?

### 🔢 2. Calcula Pearson y Spearman

Calcula ambos coeficientes. ¿Coinciden en la dirección? ¿Cuál destacarías en un informe y qué parte del gráfico respalda tu elección?

In [ ]:
# Ejemplo anterior: cor(suiza$Education, suiza$Fertility, method = "pearson")
# Ahora calcula Pearson y Spearman con el nuevo predictor.

print(c(Pearson = cor(suiza$InfantMortality, suiza$Fertility, method = "pearson"),
        Spearman = cor(suiza$InfantMortality, suiza$Fertility, method = "spearman")))
# Ambos son positivos. Si queremos resumir el orden, podemos destacar Spearman;
# si la pregunta se centra en la tendencia lineal, usamos Pearson.

### 📐 3. Ajusta e interpreta la recta

Usa la fecundidad como resultado. Reporta el R² y explica cuánto cambia la fecundidad estimada cuando la mortalidad infantil sube un punto porcentual.

In [ ]:
# Ejemplo anterior: modelo_educacion <- lm(Fertility ~ Education, data = suiza)
# Ejemplo anterior: summary(modelo_educacion)
# Ejemplo anterior: confint(modelo_educacion)
# Ahora ajusta la recta con mortalidad infantil. Lee la pendiente y el R².

modelo_infantil <- lm(Fertility ~ InfantMortality, data = suiza)
summary(modelo_infantil)
# Un punto porcentual adicional de mortalidad infantil se asocia con
# 1,79 puntos más en el índice de fecundidad ajustado. R² ≈ 0,17.

### ✍️ 4. Redacta tu conclusión

Escribe cuatro oraciones. Puedes escribirlas como comentarios de R: deja el `#` al inicio de cada línea.

In [ ]:
# Ejemplo anterior: más escolaridad se asoció con menor fecundidad estimada.
# Ahora escribe cuatro oraciones sobre mortalidad infantil y fecundidad.

# 1. Preguntamos cómo se relaciona la mortalidad infantil con la fecundidad.
# 2. La nube mostró una tendencia ascendente con dispersión.
# 3. La recta estima 1,79 puntos más de fecundidad por cada punto porcentual
#    adicional de mortalidad infantil.
# 4. Los datos son de distritos y la asociación no permite afirmar causalidad.

🎟️ **Para salir:** explica en una oración la diferencia entre correlación, pendiente y predicción. Nombra una variable que investigarías antes de plantear una explicación causal.

## 📚 Fuentes

- [Documentación de R: Swiss Fertility and Socioeconomic Indicators (1888)](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/swiss.html).